# 06 — Naïve Bayes baseline (RQ1)

Rung 2 of the baseline ladder: bag-of-words counts + Multinomial Naïve Bayes,
one binary model per construct (1A, 1B, 2). Features are fit on `train_fit`
and scored on `validation`; the held-out test set is scored once, later, in the
final build. Stop-word removal is **disabled** in the reported model, because
pronouns and in-group/out-group markers ("we", "they", "men", "feminists")
carry construct-relevant signal; a stop-word ablation is included at the end
for the appendix.

In [ ]:
# Imports and load
import pandas as pd
from pathlib import Path
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report, precision_score, recall_score, f1_score

# Repo root, whether run from notebooks/ or the root (no hard-coded local paths).
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
proc = PROJECT_ROOT / "data" / "processed"

# Fit on train_fit (1215), score on validation (405). Test stays held out.
train = pd.read_csv(proc / "train_fit.csv")
val   = pd.read_csv(proc / "validation.csv")
label_cols = ["label_1A", "label_1B", "label_2"]
for c in label_cols:
    train[c] = train[c].astype(int)
    val[c]   = val[c].astype(int)
print("train_fit:", train.shape, "| val:", val.shape)

In [ ]:
# Reported model (no stop-words):
# ---- Features: raw unigram+bigram counts, fit on TRAIN ONLY ----
# stop_words=None: keep pronouns / in-group markers, which are diagnostic here.
vectorizer = CountVectorizer(
    lowercase=True,
    ngram_range=(1, 2),   # unigrams + bigrams
    min_df=3,             # ignore terms in fewer than 3 comments (noise)
    max_df=0.9,           # ignore terms in >90% of comments (uninformative)
    stop_words=None,
)
X_train = vectorizer.fit_transform(train["text_clean"].fillna(""))  # fit on train
X_val   = vectorizer.transform(val["text_clean"].fillna(""))        # transform only on val
print("vocabulary size:", len(vectorizer.vocabulary_), "| train matrix:", X_train.shape)

# ---- Why fit_prior=False ----
# With the default (fit_prior=True) NB learns the class base rates and, under
# heavy imbalance, collapses onto the majority class (recall ~0 for positives).
# Setting fit_prior=False uses a uniform prior so positives can be recovered.
# Demonstrated on 1A:
print("\n--- 1A: default NB (fit_prior=True) ---")
print(classification_report(val["label_1A"],
      MultinomialNB().fit(X_train, train["label_1A"]).predict(X_val),
      digits=3, zero_division=0))
print("--- 1A: balanced NB (fit_prior=False) ---")
print(classification_report(val["label_1A"],
      MultinomialNB(fit_prior=False).fit(X_train, train["label_1A"]).predict(X_val),
      digits=3, zero_division=0))

# ---- All three constructs: one balanced binary model each ----
rows = []
for label in label_cols:
    model = MultinomialNB(fit_prior=False).fit(X_train, train[label])
    pred = model.predict(X_val)
    rows.append({
        "construct": label.replace("label_", ""),
        "precision": round(precision_score(val[label], pred, zero_division=0), 3),
        "recall":    round(recall_score(val[label], pred, zero_division=0), 3),
        "f1":        round(f1_score(val[label], pred, zero_division=0), 3),
        "support":   int(val[label].sum()),
    })
nb_results = pd.DataFrame(rows)
print("\n", nb_results.to_string(index=False))
nb_results.to_csv(proc / "nb_validation_metrics.csv", index=False)   # reported figures

In [ ]:
# ---- Features: raw unigram+bigram counts, fit on TRAIN ONLY ----
# stop_words=None: keep pronouns / in-group markers, which are diagnostic here.
vectorizer = CountVectorizer(
    lowercase=True,
    ngram_range=(1, 2),   # unigrams + bigrams
    min_df=3,             # ignore terms in fewer than 3 comments (noise)
    max_df=0.9,           # ignore terms in >90% of comments (uninformative)
    stop_words=None,
)
X_train = vectorizer.fit_transform(train["text_clean"].fillna(""))  # fit on train
X_val   = vectorizer.transform(val["text_clean"].fillna(""))        # transform only on val
print("vocabulary size:", len(vectorizer.vocabulary_), "| train matrix:", X_train.shape)

# ---- Why fit_prior=False ----
# With the default (fit_prior=True) NB learns the class base rates and, under
# heavy imbalance, collapses onto the majority class (recall ~0 for positives).
# Setting fit_prior=False uses a uniform prior so positives can be recovered.
# Demonstrated on 1A:
print("\n--- 1A: default NB (fit_prior=True) ---")
print(classification_report(val["label_1A"],
      MultinomialNB().fit(X_train, train["label_1A"]).predict(X_val),
      digits=3, zero_division=0))
print("--- 1A: balanced NB (fit_prior=False) ---")
print(classification_report(val["label_1A"],
      MultinomialNB(fit_prior=False).fit(X_train, train["label_1A"]).predict(X_val),
      digits=3, zero_division=0))

# ---- All three constructs: one balanced binary model each ----
rows = []
for label in label_cols:
    model = MultinomialNB(fit_prior=False).fit(X_train, train[label])
    pred = model.predict(X_val)
    rows.append({
        "construct": label.replace("label_", ""),
        "precision": round(precision_score(val[label], pred, zero_division=0), 3),
        "recall":    round(recall_score(val[label], pred, zero_division=0), 3),
        "f1":        round(f1_score(val[label], pred, zero_division=0), 3),
        "support":   int(val[label].sum()),
    })
nb_results = pd.DataFrame(rows)
print("\n", nb_results.to_string(index=False))
nb_results.to_csv(proc / "nb_validation_metrics.csv", index=False)   # reported figures

### Stop-word ablation (appendix)

The same model with English stop-words removed, for comparison only. This is
**not** the reported configuration; it is kept to show the effect of discarding
pronouns and function words. Saved to a separate file so it does not overwrite
the reported metrics above.

In [ ]:
# Identical pipeline, stop_words="english" — appendix ablation only.
vec_sw = CountVectorizer(lowercase=True, ngram_range=(1, 2), min_df=3, max_df=0.9,
                         stop_words="english")
Xtr_sw = vec_sw.fit_transform(train["text_clean"].fillna(""))
Xva_sw = vec_sw.transform(val["text_clean"].fillna(""))

rows = []
for label in label_cols:
    pred = MultinomialNB(fit_prior=False).fit(Xtr_sw, train[label]).predict(Xva_sw)
    rows.append({
        "construct": label.replace("label_", ""),
        "precision": round(precision_score(val[label], pred, zero_division=0), 3),
        "recall":    round(recall_score(val[label], pred, zero_division=0), 3),
        "f1":        round(f1_score(val[label], pred, zero_division=0), 3),
        "support":   int(val[label].sum()),
    })
nb_results_sw = pd.DataFrame(rows)
print(nb_results_sw.to_string(index=False))
nb_results_sw.to_csv(proc / "nb_validation_metrics_stopwords.csv", index=False)   # ablation